In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# -----------------------------
# Paramètres "gisement"
# -----------------------------
MU_GISEMENT = 1.27          # moyenne du gisement (%)
SIGMA_PETIT_SUPPORT = 0.8   # écart-type au petit support
REF_TONNAGE = 35.0          # tonnage de référence (t)
TC_COLOR = "#ff66cc"        # rose

# -----------------------------
# Modèle simple
# - La teneur réelle d'un camion est la moyenne de petites unités.
# - Plus le camion est gros, plus sa variabilité réelle diminue.
# - La décision est prise sur une teneur échantillonnée Z*.
# -----------------------------
def widget_biais_conditionnel(
    truck_tonnage=35,
    tc=1.30,
    sampling_noise=0.18,
    n_trucks=1500,
    seed=0
):
    rng = np.random.default_rng(seed)

    # Effet de support simplifié
    n_units = max(1, int(round(truck_tonnage)))
    sigma_camion = SIGMA_PETIT_SUPPORT / np.sqrt(n_units / REF_TONNAGE)

    # Teneurs réelles des camions
    z_reel = rng.normal(MU_GISEMENT, sigma_camion, n_trucks)

    # Teneurs échantillonnées / estimées
    z_est = z_reel + rng.normal(0, sampling_noise, n_trucks)

    # Décision sur l'estimé
    usine = z_est > tc
    sterile = ~usine

    # Erreurs de tri
    dilution = usine & (z_reel <= tc)
    perte = sterile & (z_reel > tc)

    # Statistiques
    pct_usine = 100 * usine.mean()
    pct_dilution = 100 * dilution.mean()
    pct_perte = 100 * perte.mean()

    if usine.sum() > 0:
        mean_real_usine = z_reel[usine].mean()
        mean_est_usine = z_est[usine].mean()
        surestimation_abs = mean_est_usine - mean_real_usine
        surestimation_pct = 100 * surestimation_abs / mean_real_usine if mean_real_usine != 0 else np.nan
    else:
        mean_real_usine = np.nan
        mean_est_usine = np.nan
        surestimation_abs = np.nan
        surestimation_pct = np.nan

    # Figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    fig.suptitle("Biais conditionnel avec sélectivité sur des camions", fontsize=14)

    # -----------------------------
    # 1) Nuage Z réel (y) vs Z* estimé (x)
    # -----------------------------
    ax = axes[0]
    n_show = min(1500, n_trucks)
    idx = rng.choice(n_trucks, size=n_show, replace=False)

    x = z_est[idx]
    y = z_reel[idx]
    sel = usine[idx]

    lo = min(z_reel.min(), z_est.min(), tc) - 0.25
    hi = max(z_reel.max(), z_est.max(), tc) + 0.25

    # Stérile / usine
    ax.scatter(x[~sel], y[~sel], s=10, alpha=0.22, color="lightgray", label="Stérile")
    ax.scatter(x[sel], y[sel], s=14, alpha=0.45, color="tab:blue", label="Usine")

    # Droite idéale
    ax.plot([lo, hi], [lo, hi], "--", color="black", linewidth=1.8, label="y = x")

    # Coupures
    ax.axvline(tc, color=TC_COLOR, linewidth=2.2, label="Tc estimé")
    ax.axhline(tc, color=TC_COLOR, linestyle="--", linewidth=2.2, label="Tc réel")

    # Moyenne usine + régression sur les seuls camions usine
    if usine.sum() > 2:
        ax.scatter(mean_est_usine, mean_real_usine, s=140, marker="X", color="red", label="Moyenne usine")
        a, b = np.polyfit(z_est[usine], z_reel[usine], 1)
        xx = np.linspace(tc, hi, 100)
        ax.plot(xx, a * xx + b, color="red", linewidth=2.2, label="Régression usine")

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Teneur estimée Z* (%)")
    ax.set_ylabel("Teneur réelle Z (%)")
    ax.set_title("Sélection sur Z*")
    ax.legend(fontsize=8, loc="lower right")

    # -----------------------------
    # 2) Texte synthèse
    # -----------------------------
    ax = axes[1]
    ax.axis("off")

    txt_lines = [
        f"Moyenne gisement              : {MU_GISEMENT:.2f} %",
        f"Ecart-type petit support      : {SIGMA_PETIT_SUPPORT:.2f}",
        f"Ecart-type camion            ≈ {sigma_camion:.3f}",
        "",
        f"% envoyé à l'usine           : {pct_usine:.1f} %",
        f"% dilution                   : {pct_dilution:.1f} %",
        f"% perte                      : {pct_perte:.1f} %",
        "",
        f"Teneur moyenne échantillon. usine   : {mean_est_usine:.2f} %",
        f"Teneur moyenne réelle à l'usine     : {mean_real_usine:.2f} %",  
    ]
    base_txt = "\n".join(txt_lines)

    ax.text(
        0.02, 0.98, base_txt,
        va="top", ha="left",
        fontsize=11, family="monospace"
    )

    # Ligne rouge séparée pour la sur-estimation
    if np.isfinite(surestimation_pct):
        ax.text(
            0.02, 0.42,
            f"Sur-estimation              : {surestimation_pct:.1f} %",
            va="top", ha="left",
            fontsize=11, family="monospace", color="red", fontweight="bold"
        )
    else:
        ax.text(
            0.02, 0.42,
            "Sur-estimation              : n/a",
            va="top", ha="left",
            fontsize=11, family="monospace", color="red", fontweight="bold"
        )

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()
    plt.close(fig)


title = widgets.HTML(
    "<h3>Biais conditionnel avec sélectivité</h3>"
    "<p>"
    "Des camions prélèvent un gisement de moyenne <b>1.27 %</b>. "
    "La décision usine / stérile est prise sur une teneur échantillonnée <b>Z*</b>. "
    "Quand on sélectionne les camions avec <b>Z* &gt; Tc</b>, "
    "la moyenne des échantillons des camions usine sur-estime leur moyenne réelle."
    "</p>"
)

w_truck = widgets.IntSlider(
    value=35, min=10, max=120, step=5,
    description="Camion (T)", continuous_update=False
)

w_tc = widgets.FloatSlider(
    value=1.30, min=0.0, max=2.5, step=0.02,
    description="Tc (%)", continuous_update=False
)

# laissé visible mais pas central
w_noise = widgets.FloatSlider(
    value=0.18, min=0.00, max=0.60, step=0.02,
    description="Bruit", continuous_update=False
)

w_seed = widgets.IntSlider(
    value=0, min=0, max=99, step=1,
    description="Seed", continuous_update=False
)

out = widgets.interactive_output(
    widget_biais_conditionnel,
    {
        "truck_tonnage": w_truck,
        "tc": w_tc,
        "sampling_noise": w_noise,
        "seed": w_seed
    }
)

ui = widgets.VBox([
    title,
    widgets.HBox([w_truck, w_tc, w_noise, w_seed]),
    out
])

display(ui)


Chaque camion a une teneur réelle 𝑍, mais la décision est prise sur une teneur échantillonnée 𝑍∗
Si on envoie à l’usine les camions dont  𝑍∗ >𝑇𝑐, alors la moyenne des 𝑍∗ des camions retenus surestime la moyenne réelle des camions effectivement envoyés à l’usine.
C’est un biais conditionnel créé par la sélection sur une estimation imparfaite.